In [ ]:
import openvino as ov
from pathlib import Path

core = ov.Core()
fp16_text_model_path = Path("./jina_model_path/jina-clip-text_v2_fp16.xml")
fp16_vision_model_path = Path("./jina_model_path/jina-clip-vision_v2_fp16.xml")


In [ ]:
from transformers import AutoTokenizer
from transformers import AutoImageProcessor
tokenizer_directory_path = Path("./jina_model_path")
image_processor_path = Path("./jina_model_path")
tokenizer = AutoTokenizer.from_pretrained(tokenizer_directory_path, trust_remote_code=True)
image_processor = AutoImageProcessor.from_pretrained(image_processor_path, trust_remote_code=True)


In [ ]:
text = "一只小海豹"
texts = [text] 





In [ ]:
from PIL import Image

image = "./image.png"
image = Image.open(image)
images = [image]

In [ ]:
tokenizer_kwargs = dict()
tokenizer_kwargs["padding"] = "max_length"
tokenizer_kwargs["max_length"] = 512
tokenizer_kwargs["truncation"] = True

text_inputs = tokenizer(
    texts,
    return_tensors="pt",
    **tokenizer_kwargs,
).to("cpu")

In [ ]:
vision_inputs = image_processor(images=images, return_tensors="pt")

In [ ]:
text_inputs

In [ ]:
vision_inputs

In [ ]:
text_inputs["input_ids"]

In [ ]:
vision_inputs["pixel_values"]

In [ ]:
compiled_text_model = core.compile_model(fp16_text_model_path, device_name="CPU")
compiled_vision_model = core.compile_model(fp16_vision_model_path, device_name="CPU")

In [ ]:
text_ov_res = compiled_text_model(text_inputs["input_ids"])
vis_ov_res = compiled_vision_model(vision_inputs["pixel_values"])


In [ ]:
text_ov_res

In [ ]:
vis_ov_res

In [ ]:
import numpy as np

# 提取特征向量
text_features = list(text_ov_res.values())[0]  # shape: (1, 1024)
vision_features = list(vis_ov_res.values())[0]  # shape: (1, 1024)

# L2归一化
def normalize_features(features):
    norm = np.linalg.norm(features, axis=1, keepdims=True)
    return features / norm

text_features_norm = normalize_features(text_features)
vision_features_norm = normalize_features(vision_features)

# 计算余弦相似度
similarity = np.dot(text_features_norm, vision_features_norm.T)[0, 0]

print(f"文本: {text}")
print(f"文本特征向量形状: {text_features.shape}")
print(f"图像特征向量形状: {vision_features.shape}")
print(f"余弦相似度: {similarity:.4f}")